# KOHLER AI Bathroom Designer — Complete Retrieval Pipeline

This notebook rebuilds the **retrieval layer** as a clean, self-contained pipeline.

### Pipeline

**Product JSON → Data cleaning → Structured retrieval → ChromaDB semantic retrieval → Hybrid retrieval → Candidate ranking**

The design deliberately keeps hard constraints separate from semantic similarity:

- **Structured retrieval** handles exact facts such as category, dimensions, installation, electrical requirements, and price when price data exists.
- **ChromaDB semantic retrieval** handles natural-language preferences such as *Japanese Zen*, *minimalist*, *modern*, *luxury*, and descriptive product features.
- **Hybrid retrieval** combines both without allowing semantic similarity to override hard physical constraints.

> Current catalog size is expected to be around 36 products. The same pipeline can be reused when the catalog grows to 100–120 products.


In [2]:
# Cell 2 — Imports

import json
import math
from pathlib import Path

import chromadb
import pandas as pd

print("ChromaDB version:", chromadb.__version__)
print("Pandas version:", pd.__version__)
print("Imports successful.")


ChromaDB version: 1.5.9
Pandas version: 3.0.5
Imports successful.


In [3]:
# Cell 3 — Configuration

JSON_PATH = Path(
    r"C:\Users\Abhist\Desktop\KOHLER\DATA\kohler_ai_bathroom.products.json"
)

CHROMA_PATH = Path(
    r"C:\Users\Abhist\Desktop\KOHLER\data\chroma_db"
)

COLLECTION_NAME = "kohler_products"

print("JSON path:", JSON_PATH)
print("JSON exists:", JSON_PATH.exists())
print("Chroma path:", CHROMA_PATH)


JSON path: C:\Users\Abhist\Desktop\KOHLER\DATA\kohler_ai_bathroom.products.json
JSON exists: True
Chroma path: C:\Users\Abhist\Desktop\KOHLER\data\chroma_db


In [4]:
# Cell 4 — Load product JSON

if not JSON_PATH.exists():
    raise FileNotFoundError(
        f"Product JSON was not found at:\n{JSON_PATH}\n\n"
        "Check the JSON_PATH in Cell 3."
    )

with open(JSON_PATH, "r", encoding="utf-8") as f:
    products = json.load(f)

if not isinstance(products, list):
    raise ValueError(
        "Expected the JSON root to contain a list of products."
    )

print("Products loaded:", len(products))
print("Data type:", type(products).__name__)

if len(products) == 0:
    raise ValueError("The product JSON contains no products.")


Products loaded: 36
Data type: list


In [5]:
# Cell 5 — Inspect one product

print(json.dumps(products[0], indent=4, ensure_ascii=False))


{
    "_id": {
        "$oid": "6aa96a2826ad3942f37da9fc"
    },
    "product_id": "1408991-IN4-A",
    "category": "Other",
    "collection": "Not specified",
    "color": [
        "White"
    ],
    "dimensions": {
        "width_mm": 344,
        "width_mm_available": true,
        "depth_mm": 483,
        "depth_mm_available": true,
        "height_mm": 141,
        "height_mm_available": true
    },
    "electrical": {
        "required": false,
        "required_source": "source",
        "voltage": "Not specified",
        "power_w": -1,
        "power_available": false
    },
    "extraction_method": "Groq text extraction",
    "extraction_status": "success",
    "features": [
        "Ergonomic and Straight line design suitable for compact rooms with deck space.",
        "Easy installation as it requires counter cutting to accommodate drain only.",
        "Optimum Depth to contain Splashes.",
        "Above counter without faucet deck.",
        "Only drain cutting (no prof

In [6]:
# Cell 6 — Flatten product data

df = pd.json_normalize(products)

print("Number of products:", len(df))
print("Number of fields:", len(df.columns))

display(df.head())


Number of products: 36
Number of fields: 36


,product_id,category,collection,color,extraction_method,extraction_status,features,material,product_name,source_pdf,...,dimensions.height_mm_available,electrical.required,electrical.required_source,electrical.voltage,electrical.power_w,electrical.power_available,installation.type,installation.waste_outlet,installation.rough_in_mm,installation.rough_in_available
0,1408991-IN4-A,Other,Not specified,[White],Groq text extraction,success,[Ergonomic and Straight line design suitable f...,Vitreous China,SPAN® Square Vessel Without Deck ( Small),1408991-IN4.pdf,...,True,False,source,Not specified,-1,False,Above Counter,Not specified,-1,False
1,29024IN-1,Vessel,Not specified,[White],Groq text extraction,success,"[Vitreous China, Circular Design, Without Over...",Vitreous China,CHALICE ROUND VESSEL 1TAP HOLE,29024IN-1_Chalice w deck vessel.pdf,...,True,False,source,Not specified,-1,False,Above counter,Not specified,-1,False
2,1311520-A04-D,BASINS / COUNTERTOP,Not specified,"[white, honed black]",Groq text extraction,success,"[Bench top installation, Slim vessel, CleanCoa...",Ceramic,Mica® Square 393mm Basin,90011T.pdf,...,False,False,source,Not specified,-1,False,Bench top installation,DN32 waste without overflow,50,True
3,K-12927IN,Other,Not specified,"[Polished Chrome, Vibrant® French Gold, Vibran...",Groq text extraction,success,[Designed to coordinate with a range of KOHLER...,Not specified,Complimentary™ Hygiene Spray,K-12927IN_spec_IN_Kohler_en.pdf,...,False,False,source,Not specified,-1,False,Wall-mount,Not specified,-1,False
4,K-1381T-S,Toilet,Veil,[White],Groq text extraction,success,[One-piece toilets integrate the tank and bowl...,Plastic seat,Veil™ One-piece elongated toilet with skirted ...,K-1381T-S_spec_IN_Kohler_en.pdf,...,True,False,source,Not specified,-1,False,Floor-mount,Floor,305,True


In [7]:
# Cell 7 — Create retrieval dataset

retrieval_df = df.copy()

# Make sure important columns exist.
required_columns = [
    "product_id",
    "product_name",
    "category",
    "subcategory",
    "collection",
    "material",
    "color",
    "features",
]

for column in required_columns:
    if column not in retrieval_df.columns:
        retrieval_df[column] = pd.NA

print("Retrieval dataset created.")
print("Products:", len(retrieval_df))


Retrieval dataset created.
Products: 36


In [8]:
# Cell 8 — Normalize categories

def normalize_category(category):
    """Convert inconsistent source categories into stable retrieval categories."""

    if category is None:
        return "Other"

    # Avoid treating pandas missing values as booleans.
    try:
        if pd.isna(category):
            return "Other"
    except (TypeError, ValueError):
        pass

    value = str(category).strip().lower()

    if not value or value in {"none", "nan", "not specified"}:
        return "Other"

    # Check specific categories before broader categories.
    if "toilet seat" in value:
        return "Toilet Seat"

    if "toilet" in value:
        return "Toilet"

    if "faucet trim" in value:
        return "Faucet Trim"

    if "faucet" in value:
        return "Faucet"

    if "shower" in value:
        return "Shower"

    if "vanity" in value:
        return "Vanity"

    if "bathtub" in value or "bath tub" in value:
        return "Bathtub"

    if "mirror" in value or "cabinet" in value:
        return "Mirror"

    if "sink" in value or "basin" in value or "vessel" in value:
        return "Sink"

    return "Other"


retrieval_df["category_normalized"] = (
    retrieval_df["category"].apply(normalize_category)
)

print("Normalized categories:")
display(retrieval_df["category_normalized"].value_counts())


Normalized categories:


category_normalized
Faucet         15
Sink           10
Toilet          5
Other           4
Toilet Seat     1
Faucet Trim     1
Name: count, dtype: int64

In [9]:
# Cell 9 — Clean numeric fields

numeric_fields = [
    "dimensions.width_mm",
    "dimensions.depth_mm",
    "dimensions.height_mm",
    "dimensions.width_mm_available",
    "dimensions.depth_mm_available",
    "dimensions.height_mm_available",
    "electrical.power_w",
    "installation.rough_in_mm",
    "installation.rough_in_available",
]

for field in numeric_fields:
    if field in retrieval_df.columns:
        retrieval_df[field] = pd.to_numeric(
            retrieval_df[field],
            errors="coerce"
        )

# Negative measurements/power values are invalid for retrieval.
non_negative_fields = [
    "dimensions.width_mm",
    "dimensions.depth_mm",
    "dimensions.height_mm",
    "electrical.power_w",
    "installation.rough_in_mm",
]

for field in non_negative_fields:
    if field in retrieval_df.columns:
        retrieval_df.loc[
            retrieval_df[field] < 0,
            field
        ] = pd.NA

print("Numeric fields cleaned.")


Numeric fields cleaned.


In [10]:
# Cell 10 — Prepare price field

# The current extracted catalog may not contain product prices.
# We keep the field explicitly so budget filtering can be added
# automatically when price data becomes available.

if "price_inr" in retrieval_df.columns:
    retrieval_df["price_inr"] = pd.to_numeric(
        retrieval_df["price_inr"],
        errors="coerce"
    )
elif "price" in retrieval_df.columns:
    retrieval_df["price_inr"] = pd.to_numeric(
        retrieval_df["price"],
        errors="coerce"
    )
else:
    retrieval_df["price_inr"] = pd.NA

print("Products with known price:", retrieval_df["price_inr"].notna().sum())
print("Products without price:", retrieval_df["price_inr"].isna().sum())

if retrieval_df["price_inr"].notna().sum() == 0:
    print(
        "\nNOTE: No price data is currently present in the product JSON. "
        "Budget filtering will therefore return UNKNOWN for price until "
        "prices are added to the catalog."
    )


Products with known price: 0
Products without price: 36

NOTE: No price data is currently present in the product JSON. Budget filtering will therefore return UNKNOWN for price until prices are added to the catalog.


In [11]:
# Cell 11 — Safe value helpers

def is_missing(value):
    """Safely determine whether a scalar value is missing."""
    if value is None:
        return True

    try:
        result = pd.isna(value)
        return bool(result) if not hasattr(result, "__len__") else False
    except (TypeError, ValueError):
        return False


def clean_text(value):
    """Convert a scalar value to clean text or None."""
    if is_missing(value):
        return None

    value = str(value).strip()

    if not value or value.lower() in {"nan", "none", "not specified"}:
        return None

    return value


def clean_list(value):
    """Convert list/string values into a clean list of strings."""
    if is_missing(value):
        return []

    if isinstance(value, list):
        result = []
        for item in value:
            item = clean_text(item)
            if item and item not in result:
                result.append(item)
        return result

    item = clean_text(value)
    return [item] if item else []


print("Helper functions ready.")


Helper functions ready.


In [12]:
# Cell 12 — Structured constraint helpers

def apply_max_constraint(dataframe, field, maximum):
    """
    Keep:
      - known value <= maximum
      - missing value

    Reject:
      - known value > maximum

    Missing values remain candidates because they need later validation
    rather than being silently treated as a failure.
    """
    if maximum is None or field not in dataframe.columns:
        return dataframe

    values = pd.to_numeric(dataframe[field], errors="coerce")

    mask = values.isna() | (values <= float(maximum))

    return dataframe[mask]


def apply_budget_constraint(dataframe, max_price_inr=None):
    """Apply a maximum price constraint while preserving unknown prices."""
    if max_price_inr is None or "price_inr" not in dataframe.columns:
        return dataframe

    prices = pd.to_numeric(
        dataframe["price_inr"],
        errors="coerce"
    )

    mask = prices.isna() | (prices <= float(max_price_inr))

    return dataframe[mask]


def apply_installation_constraint(dataframe, installation_type=None):
    """Apply installation type while retaining unknown installation data."""
    if installation_type is None:
        return dataframe

    field = "installation.type"

    if field not in dataframe.columns:
        return dataframe

    requested = str(installation_type).strip().lower()

    values = (
        dataframe[field]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    unknown = values.isin({"", "none", "nan", "not specified"})

    return dataframe[
        values.eq(requested) | unknown
    ]


def apply_rough_in_constraint(dataframe, required_rough_in_mm=None, tolerance_mm=10):
    """Apply rough-in compatibility with a configurable tolerance."""
    if required_rough_in_mm is None:
        return dataframe

    field = "installation.rough_in_mm"

    if field not in dataframe.columns:
        return dataframe

    values = pd.to_numeric(
        dataframe[field],
        errors="coerce"
    )

    compatible = (
        values.sub(float(required_rough_in_mm)).abs()
        <= float(tolerance_mm)
    )

    return dataframe[
        values.isna() | compatible
    ]


def apply_electrical_constraint(dataframe, electrical_required=None):
    """Apply electrical requirement while preserving unknown values."""
    if electrical_required is None:
        return dataframe

    field = "electrical.required"

    if field not in dataframe.columns:
        return dataframe

    values = dataframe[field].astype("object")

    normalized = (
        values
        .astype(str)
        .str.strip()
        .str.lower()
    )

    true_values = normalized.isin({"true", "yes", "1"})
    false_values = normalized.isin({"false", "no", "0"})
    unknown = ~(true_values | false_values)

    if bool(electrical_required):
        mask = true_values | unknown
    else:
        mask = false_values | unknown

    return dataframe[mask]


In [13]:
# Cell 13 — Main structured retrieval function

def filter_products(
    dataframe,
    category=None,
    max_price_inr=None,
    max_width_mm=None,
    max_depth_mm=None,
    max_height_mm=None,
    installation_type=None,
    required_rough_in_mm=None,
    electrical_required=None,
):
    """
    Retrieve products using exact structured constraints.

    Hard constraints:
      category
      budget
      width
      depth
      height
      installation type
      rough-in
      electrical requirement

    Missing source values are retained as candidates and are handled
    later by validation.
    """

    result = dataframe.copy()

    if category is not None:
        requested = str(category).strip().lower()

        if "category_normalized" in result.columns:
            categories = (
                result["category_normalized"]
                .fillna("Other")
                .astype(str)
                .str.strip()
                .str.lower()
            )

            result = result[categories.eq(requested)]

    result = apply_budget_constraint(
        result,
        max_price_inr
    )

    result = apply_max_constraint(
        result,
        "dimensions.width_mm",
        max_width_mm
    )

    result = apply_max_constraint(
        result,
        "dimensions.depth_mm",
        max_depth_mm
    )

    result = apply_max_constraint(
        result,
        "dimensions.height_mm",
        max_height_mm
    )

    result = apply_installation_constraint(
        result,
        installation_type
    )

    result = apply_rough_in_constraint(
        result,
        required_rough_in_mm
    )

    result = apply_electrical_constraint(
        result,
        electrical_required
    )

    return result.reset_index(drop=True)


print("Structured retrieval function ready.")


Structured retrieval function ready.


In [14]:
# Cell 14 — Test structured retrieval

structured_results = filter_products(
    retrieval_df,
    category="Toilet",
    max_depth_mm=700,
    max_height_mm=500,
    installation_type="Floor-mount",
    required_rough_in_mm=305,
)

print("Structured candidates:", len(structured_results))

display(
    structured_results[
        [
            "product_id",
            "product_name",
            "category_normalized",
            "price_inr",
            "dimensions.width_mm",
            "dimensions.depth_mm",
            "dimensions.height_mm",
        ]
    ]
)


Structured candidates: 3


,product_id,product_name,category_normalized,price_inr,dimensions.width_mm,dimensions.depth_mm,dimensions.height_mm
0,K-1381T-S,Veil™ One-piece elongated toilet with skirted ...,Toilet,<NA>,NaN,NaN,390.0
1,K-28529IN,"Leap™ One-piece round-front smart toilet, dual...",Toilet,<NA>,NaN,NaN,400.0
2,K-3983IN-S,Reach™ One-piece round-front toilet with skirt...,Toilet,<NA>,NaN,NaN,391.0


In [15]:
# Cell 15 — Product validation

def check_max_value(value, maximum):
    if maximum is None:
        return True
    if is_missing(value):
        return None
    return float(value) <= float(maximum)


def check_rough_in(value, required, tolerance_mm=10):
    if required is None:
        return True
    if is_missing(value):
        return None
    return abs(float(value) - float(required)) <= tolerance_mm


def validate_product(
    product,
    max_price_inr=None,
    max_width_mm=None,
    max_depth_mm=None,
    max_height_mm=None,
    required_rough_in_mm=None,
    installation_type=None,
    electrical_required=None,
):
    """Return PASS / FAIL / UNKNOWN-compatible boolean checks."""

    checks = {}

    # Budget
    if max_price_inr is not None:
        price = product.get("price_inr")

        if is_missing(price):
            checks["budget"] = None
        else:
            checks["budget"] = float(price) <= float(max_price_inr)

    # Dimensions
    checks["width"] = check_max_value(
        product.get("dimensions.width_mm"),
        max_width_mm
    )

    checks["depth"] = check_max_value(
        product.get("dimensions.depth_mm"),
        max_depth_mm
    )

    checks["height"] = check_max_value(
        product.get("dimensions.height_mm"),
        max_height_mm
    )

    # Rough-in
    checks["rough_in"] = check_rough_in(
        product.get("installation.rough_in_mm"),
        required_rough_in_mm
    )

    # Installation
    if installation_type is not None:
        actual = clean_text(
            product.get("installation.type")
        )

        if actual is None:
            checks["installation"] = None
        else:
            checks["installation"] = (
                actual.strip().lower()
                == str(installation_type).strip().lower()
            )

    # Electrical
    if electrical_required is not None:
        value = product.get("electrical.required")

        if is_missing(value):
            checks["electrical"] = None
        else:
            normalized = str(value).strip().lower()

            if normalized in {"true", "yes", "1"}:
                actual = True
            elif normalized in {"false", "no", "0"}:
                actual = False
            else:
                actual = None

            checks["electrical"] = (
                None
                if actual is None
                else actual == bool(electrical_required)
            )

    return checks


def summarize_checks(checks):
    result = {}

    for constraint, value in checks.items():
        if value is True:
            result[constraint] = "PASS"
        elif value is False:
            result[constraint] = "FAIL"
        else:
            result[constraint] = "UNKNOWN"

    return result


In [16]:
# Cell 16 — Create validation report

def create_validation_report(dataframe, **requirements):
    reports = []

    for _, product in dataframe.iterrows():
        checks = validate_product(
            product,
            **requirements
        )

        summary = summarize_checks(checks)

        report = {
            "product_id": product.get("product_id"),
            "product_name": product.get("product_name"),
            "category": product.get("category_normalized"),
        }

        for constraint, status in summary.items():
            report[f"{constraint}_status"] = status

        reports.append(report)

    return pd.DataFrame(reports)


validation_report = create_validation_report(
    retrieval_df,
    max_depth_mm=700,
    max_height_mm=500,
    required_rough_in_mm=305,
    installation_type="Floor-mount",
)

display(validation_report.head(10))


,product_id,product_name,category,width_status,depth_status,height_status,rough_in_status,installation_status
0,1408991-IN4-A,SPAN® Square Vessel Without Deck ( Small),Other,PASS,PASS,PASS,UNKNOWN,FAIL
1,29024IN-1,CHALICE ROUND VESSEL 1TAP HOLE,Sink,PASS,PASS,PASS,UNKNOWN,FAIL
2,1311520-A04-D,Mica® Square 393mm Basin,Sink,PASS,PASS,UNKNOWN,FAIL,FAIL
3,K-12927IN,Complimentary™ Hygiene Spray,Other,PASS,UNKNOWN,UNKNOWN,UNKNOWN,FAIL
4,K-1381T-S,Veil™ One-piece elongated toilet with skirted ...,Toilet,PASS,UNKNOWN,PASS,PASS,PASS
5,K-17629T-NS,Ove™ One-piece round-front toilet with skirted...,Toilet,PASS,UNKNOWN,PASS,FAIL,PASS
6,K-17660T-M,Ove™ Quiet-Close™ elongated toilet seat,Toilet Seat,PASS,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN
7,K-1851IN,Brive Plus,Other,PASS,UNKNOWN,PASS,FAIL,PASS
8,K-1853IN,Brive Plus,Other,PASS,UNKNOWN,PASS,FAIL,PASS
9,K-21226IN,ModernLife Edge 600 mm rectangular vessel bath...,Sink,PASS,PASS,UNKNOWN,UNKNOWN,FAIL


## Semantic Retrieval with ChromaDB

The semantic layer uses product descriptions rather than hard constraints.

Examples of semantic concepts:

- Japanese Zen
- minimalist
- modern
- luxury
- elegant
- clean lines
- calm aesthetic
- water-efficient
- smart bathroom

Dimensions, budget, and other hard constraints are **not trusted to semantic similarity**.


In [17]:
# Cell 17 — Build semantic text

def create_semantic_text(product):
    """Create a robust natural-language representation for embeddings."""

    parts = []

    product_name = clean_text(product.get("product_name"))
    category = clean_text(product.get("category_normalized"))
    subcategory = clean_text(product.get("subcategory"))
    collection = clean_text(product.get("collection"))
    material = clean_text(product.get("material"))
    installation = clean_text(product.get("installation.type"))

    if product_name:
        parts.append(f"Product: {product_name}")

    if category:
        parts.append(f"Category: {category}")

    if subcategory:
        parts.append(f"Subcategory: {subcategory}")

    if collection:
        parts.append(f"Collection: {collection}")

    if material:
        parts.append(f"Material: {material}")

    colors = clean_list(product.get("color"))
    if colors:
        parts.append("Color: " + ", ".join(colors))

    features = clean_list(product.get("features"))
    if features:
        parts.append("Features: " + ", ".join(features))

    if installation:
        parts.append(f"Installation: {installation}")

    # Include useful electrical information when explicitly known.
    electrical_required = product.get("electrical.required")
    if not is_missing(electrical_required):
        parts.append(
            f"Electrical required: {str(electrical_required)}"
        )

    return ". ".join(parts)


retrieval_df["semantic_text"] = retrieval_df.apply(
    create_semantic_text,
    axis=1
)

# Guarantee that Chroma never receives an empty document.
retrieval_df["semantic_text"] = retrieval_df["semantic_text"].apply(
    lambda x: x if clean_text(x) else "KOHLER bathroom product."
)

print("Semantic text created.")
display(
    retrieval_df[
        [
            "product_id",
            "product_name",
            "category_normalized",
            "semantic_text",
        ]
    ].head(5)
)


Semantic text created.


,product_id,product_name,category_normalized,semantic_text
0,1408991-IN4-A,SPAN® Square Vessel Without Deck ( Small),Other,Product: SPAN® Square Vessel Without Deck ( Sm...
1,29024IN-1,CHALICE ROUND VESSEL 1TAP HOLE,Sink,Product: CHALICE ROUND VESSEL 1TAP HOLE. Categ...
2,1311520-A04-D,Mica® Square 393mm Basin,Sink,Product: Mica® Square 393mm Basin. Category: S...
3,K-12927IN,Complimentary™ Hygiene Spray,Other,Product: Complimentary™ Hygiene Spray. Categor...
4,K-1381T-S,Veil™ One-piece elongated toilet with skirted ...,Toilet,Product: Veil™ One-piece elongated toilet with...


In [18]:
# Cell 18 — Initialize ChromaDB

CHROMA_PATH.mkdir(
    parents=True,
    exist_ok=True
)

chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_PATH)
)

print("ChromaDB initialized.")
print("Database path:", CHROMA_PATH)


ChromaDB initialized.
Database path: C:\Users\Abhist\Desktop\KOHLER\data\chroma_db


In [19]:
# Cell 19 — Initialize embedding function

from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

embedding_function = DefaultEmbeddingFunction()

print("Default embedding function initialized.")


Default embedding function initialized.


In [20]:
# Cell 20 — Test embeddings

test_text = [
    "Japanese Zen minimalist bathroom faucet"
]

test_embedding = embedding_function(test_text)

print("Embedding generated successfully.")
print("Number of embeddings:", len(test_embedding))

if len(test_embedding) > 0:
    print("Embedding dimension:", len(test_embedding[0]))


Embedding generated successfully.
Number of embeddings: 1
Embedding dimension: 384


In [21]:
# Cell 21 — Create / reuse Chroma collection

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function,
    metadata={
        "description": "KOHLER bathroom product semantic retrieval"
    },
)

print("Collection:", collection.name)
print("Existing records:", collection.count())


Collection: kohler_products
Existing records: 0


In [22]:
# Cell 22 — Prepare Chroma metadata

def chroma_metadata_value(value):
    """Convert a value to a Chroma-safe scalar."""

    if is_missing(value):
        return "Not specified"

    if isinstance(value, list):
        cleaned = clean_list(value)
        return ", ".join(cleaned) if cleaned else "Not specified"

    return value


chroma_ids = []
chroma_documents = []
chroma_metadatas = []

for _, product in retrieval_df.iterrows():

    product_id = clean_text(product.get("product_id"))

    if not product_id:
        # Product IDs are required for stable retrieval and hybrid filtering.
        continue

    chroma_ids.append(product_id)

    chroma_documents.append(
        product["semantic_text"]
    )

    chroma_metadatas.append({
        "product_id": product_id,
        "product_name": chroma_metadata_value(
            product.get("product_name")
        ),
        "category": chroma_metadata_value(
            product.get("category_normalized")
        ),
        "collection": chroma_metadata_value(
            product.get("collection")
        ),
        "material": chroma_metadata_value(
            product.get("material")
        ),
        "color": chroma_metadata_value(
            product.get("color")
        ),
        "installation_type": chroma_metadata_value(
            product.get("installation.type")
        ),
    })


print("Chroma records prepared.")
print("IDs:", len(chroma_ids))
print("Documents:", len(chroma_documents))
print("Metadata:", len(chroma_metadatas))


Chroma records prepared.
IDs: 36
Documents: 36
Metadata: 36


In [23]:
# Cell 23 — Upsert products into ChromaDB

if not chroma_ids:
    raise ValueError("No valid product IDs were available for ChromaDB.")

collection.upsert(
    ids=chroma_ids,
    documents=chroma_documents,
    metadatas=chroma_metadatas,
)

print("Products upserted successfully.")
print("ChromaDB record count:", collection.count())


Products upserted successfully.
ChromaDB record count: 36


In [24]:
# Cell 24 — Basic semantic retrieval

def semantic_search(
    query,
    n_results=5,
    category=None,
):
    """
    Search products semantically.

    Optional category is applied as a Chroma metadata filter.
    """

    query = clean_text(query)

    if not query:
        raise ValueError("Semantic query cannot be empty.")

    n_results = max(1, int(n_results))

    kwargs = {
        "query_texts": [query],
        "n_results": n_results,
    }

    if category is not None:
        kwargs["where"] = {
            "category": str(category)
        }

    return collection.query(**kwargs)


semantic_results = semantic_search(
    "Japanese Zen minimalist calm elegant bathroom fixture",
    n_results=5,
)

print("Semantic results:")
for rank, product_id in enumerate(
    semantic_results["ids"][0],
    start=1
):
    print(f"{rank}. {product_id}")


Semantic results:
1. K-1851IN
2. K-1853IN
3. K-1381T-S
4. K-33123IN-HC
5. K-5684IN-4ND


In [25]:
# Cell 25 — Convert Chroma results into a DataFrame

def semantic_results_to_dataframe(results):
    """Convert Chroma query output into a readable DataFrame."""

    ids = results.get("ids", [[]])[0]
    documents = results.get("documents", [[]])[0]
    distances = results.get("distances", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]

    rows = []

    for i, product_id in enumerate(ids):
        metadata = metadatas[i] if i < len(metadatas) else {}

        rows.append({
            "product_id": product_id,
            "semantic_distance": (
                distances[i]
                if i < len(distances)
                else None
            ),
            "product_name": metadata.get(
                "product_name",
                "Not specified"
            ),
            "category": metadata.get(
                "category",
                "Not specified"
            ),
            "collection": metadata.get(
                "collection",
                "Not specified"
            ),
            "document": (
                documents[i]
                if i < len(documents)
                else ""
            ),
        })

    return pd.DataFrame(rows)


semantic_df = semantic_results_to_dataframe(
    semantic_results
)

display(semantic_df)


,product_id,semantic_distance,product_name,category,collection,document
0,K-1851IN,0.869293,Brive Plus,Other,Not specified,Product: Brive Plus. Category: Other. Color: W...
1,K-1853IN,1.136419,Brive Plus,Other,Not specified,Product: Brive Plus. Category: Other. Color: W...
2,K-1381T-S,1.138095,Veil™ One-piece elongated toilet with skirted ...,Toilet,Veil,Product: Veil™ One-piece elongated toilet with...
3,K-33123IN-HC,1.156153,KOHLER VIVE Hidden cord one-piece round-front ...,Toilet,Vive,Product: KOHLER VIVE Hidden cord one-piece rou...
4,K-5684IN-4ND,1.160425,"Aleo+™ Wall-mount bathroom sink faucet trim, 9...",Faucet Trim,Aleo+,Product: Aleo+™ Wall-mount bathroom sink fauce...


## Hybrid Retrieval

This is the key retrieval component.

The system first applies **structured constraints**. Only products that survive those constraints are allowed into the semantic stage.

Then ChromaDB ranks those eligible products by semantic similarity.

So the logic is:

**Relevant → Allowed → Feasible → Ranked**

Semantic similarity cannot make an oversized or otherwise disallowed product eligible again.


In [26]:
# Cell 26 — Hybrid retrieval function

def hybrid_retrieval(
    dataframe,
    semantic_query,
    category=None,
    max_price_inr=None,
    max_width_mm=None,
    max_depth_mm=None,
    max_height_mm=None,
    installation_type=None,
    required_rough_in_mm=None,
    electrical_required=None,
    top_k=5,
    semantic_pool_size=20,
):
    """
    Hybrid retrieval:

    1. Structured filtering creates the eligible candidate pool.
    2. ChromaDB performs semantic retrieval only within that pool.
    3. Results are joined back to the complete product records.

    Returns:
        hybrid_df, structured_candidates
    """

    # ---------------------------------------
    # Step 1: exact structured filtering
    # ---------------------------------------
    structured_candidates = filter_products(
        dataframe,
        category=category,
        max_price_inr=max_price_inr,
        max_width_mm=max_width_mm,
        max_depth_mm=max_depth_mm,
        max_height_mm=max_height_mm,
        installation_type=installation_type,
        required_rough_in_mm=required_rough_in_mm,
        electrical_required=electrical_required,
    )

    if structured_candidates.empty:
        return pd.DataFrame(), structured_candidates

    candidate_ids = [
        str(x)
        for x in structured_candidates["product_id"]
        if not is_missing(x)
    ]

    if not candidate_ids:
        return pd.DataFrame(), structured_candidates

    # ---------------------------------------
    # Step 2: semantic retrieval restricted
    # to structured candidates
    # ---------------------------------------
    semantic_query = clean_text(semantic_query)

    if not semantic_query:
        raise ValueError("semantic_query cannot be empty.")

    semantic_pool_size = max(
        int(top_k),
        int(semantic_pool_size)
    )

    # Chroma supports metadata $in filtering.
    # If a Chroma version rejects the filter, use a larger
    # unrestricted search as a safe fallback and intersect IDs.
    try:
        results = collection.query(
            query_texts=[semantic_query],
            n_results=min(
                semantic_pool_size,
                len(candidate_ids)
            ),
            where={
                "product_id": {
                    "$in": candidate_ids
                }
            },
        )

        result_ids = results["ids"][0]
        result_distances = results.get(
            "distances",
            [[]]
        )[0]

    except Exception as exc:
        print(
            "Chroma metadata-filter query failed; "
            "using fallback intersection."
        )
        print("Reason:", exc)

        fallback_n = min(
            max(50, semantic_pool_size),
            collection.count()
        )

        results = collection.query(
            query_texts=[semantic_query],
            n_results=fallback_n,
        )

        all_ids = results["ids"][0]
        all_distances = results.get(
            "distances",
            [[]]
        )[0]

        candidate_set = set(candidate_ids)

        filtered_pairs = [
            (pid, dist)
            for pid, dist in zip(
                all_ids,
                all_distances
            )
            if pid in candidate_set
        ]

        filtered_pairs = filtered_pairs[:top_k]

        result_ids = [
            pid for pid, _ in filtered_pairs
        ]

        result_distances = [
            dist for _, dist in filtered_pairs
        ]

    if not result_ids:
        return pd.DataFrame(), structured_candidates

    # ---------------------------------------
    # Step 3: join semantic ranking back to
    # complete structured product records
    # ---------------------------------------
    lookup = structured_candidates.copy()

    lookup["product_id"] = lookup[
        "product_id"
    ].astype(str)

    result_rows = []

    for rank, (product_id, distance) in enumerate(
        zip(result_ids, result_distances),
        start=1
    ):
        matches = lookup[
            lookup["product_id"] == str(product_id)
        ]

        if matches.empty:
            continue

        row = matches.iloc[0].copy()

        row["semantic_rank"] = rank
        row["semantic_distance"] = float(distance)

        # Lower distance = more semantically similar.
        row["semantic_score"] = 1.0 / (
            1.0 + float(distance)
        )

        result_rows.append(row)

        if len(result_rows) >= top_k:
            break

    if not result_rows:
        return pd.DataFrame(), structured_candidates

    hybrid_df = pd.DataFrame(result_rows)

    return hybrid_df.reset_index(drop=True), structured_candidates.reset_index(drop=True)


print("Hybrid retrieval function ready.")


Hybrid retrieval function ready.


In [27]:
# Cell 27 — Test hybrid retrieval

hybrid_results, eligible_candidates = hybrid_retrieval(
    retrieval_df,

    semantic_query=(
        "Japanese Zen minimalist calm elegant "
        "bathroom fixture with a clean design"
    ),

    category="Toilet",

    max_depth_mm=700,
    max_height_mm=500,

    installation_type="Floor-mount",
    required_rough_in_mm=305,

    top_k=5,
)

print("Structured eligible candidates:", len(eligible_candidates))
print("Hybrid results:", len(hybrid_results))

if not hybrid_results.empty:
    display(
        hybrid_results[
            [
                "product_id",
                "product_name",
                "category_normalized",
                "collection",
                "semantic_rank",
                "semantic_distance",
                "semantic_score",
                "dimensions.width_mm",
                "dimensions.depth_mm",
                "dimensions.height_mm",
            ]
        ]
    )
else:
    print(
        "No hybrid results were found for this exact test. "
        "Try relaxing the physical constraints or changing the category."
    )


Structured eligible candidates: 3
Hybrid results: 3


,product_id,product_name,category_normalized,collection,semantic_rank,semantic_distance,semantic_score,dimensions.width_mm,dimensions.depth_mm,dimensions.height_mm
0,K-1381T-S,Veil™ One-piece elongated toilet with skirted ...,Toilet,Veil,1,1.105111,0.475034,NaN,NaN,390.0
1,K-28529IN,"Leap™ One-piece round-front smart toilet, dual...",Toilet,Not specified,2,1.166507,0.461572,NaN,NaN,400.0
2,K-3983IN-S,Reach™ One-piece round-front toilet with skirt...,Toilet,Reach™,3,1.238431,0.446741,NaN,NaN,391.0


In [28]:
# Cell 28 — Generic hybrid search helper for future app use

def search_bathroom_products(
    user_query,
    category=None,
    max_price_inr=None,
    max_width_mm=None,
    max_depth_mm=None,
    max_height_mm=None,
    installation_type=None,
    required_rough_in_mm=None,
    electrical_required=None,
    top_k=5,
):
    """
    Main function that the future Gradio application can call.

    Example:

        search_bathroom_products(
            user_query="minimalist Japanese Zen faucet",
            category="Faucet",
            max_width_mm=150,
            max_depth_mm=200,
            top_k=5
        )
    """

    results, eligible = hybrid_retrieval(
        retrieval_df,
        semantic_query=user_query,
        category=category,
        max_price_inr=max_price_inr,
        max_width_mm=max_width_mm,
        max_depth_mm=max_depth_mm,
        max_height_mm=max_height_mm,
        installation_type=installation_type,
        required_rough_in_mm=required_rough_in_mm,
        electrical_required=electrical_required,
        top_k=top_k,
    )

    return {
        "eligible_candidates": eligible,
        "results": results,
    }


print("Application-ready retrieval function created.")


Application-ready retrieval function created.


In [29]:
# Cell 29 — Example application query

search_output = search_bathroom_products(
    user_query=(
        "modern minimalist bathroom fixture, "
        "clean elegant design"
    ),
    category="Faucet",
    max_width_mm=200,
    max_depth_mm=250,
    top_k=5,
)

print(
    "Eligible candidates:",
    len(search_output["eligible_candidates"])
)

print(
    "Final semantic results:",
    len(search_output["results"])
)

if not search_output["results"].empty:
    display(
        search_output["results"][
            [
                "product_id",
                "product_name",
                "category_normalized",
                "collection",
                "semantic_score",
            ]
        ]
    )


Eligible candidates: 15
Final semantic results: 5


,product_id,product_name,category_normalized,collection,semantic_score
0,K-77364IN,"Health faucet, 8.5 lpm",Faucet,Luxe™,0.439533
1,K-98100IN-ZZ,"Beam™ Health faucet, 10 lpm",Faucet,Not specified,0.439520
2,K-23967IN-4ND,KOHLER VIVE® Tall single-handle bathroom sink ...,Faucet,VIVE,0.426061
3,K-97347T-4,Avid™ Tall Single-handle bathroom sink faucet,Faucet,Avid™ Tall,0.425062
4,K-72337IN-4ND,Aleo+™ Tall single-handle bathroom sink faucet...,Faucet,Aleo+,0.423943


## Retrieval layer completed

The notebook now provides:

1. **Product JSON loading**
2. **DataFrame normalization**
3. **Category normalization**
4. **Numeric cleanup**
5. **Budget handling**
6. **Structured dimension filtering**
7. **Installation filtering**
8. **Rough-in filtering**
9. **Electrical filtering**
10. **PASS / FAIL / UNKNOWN validation**
11. **Semantic text generation**
12. **ChromaDB persistence**
13. **Embedding generation**
14. **Semantic retrieval**
15. **Structured + semantic hybrid retrieval**
16. **Application-ready search function**

### Important limitation in the current catalog

If `price_inr` is missing for all products, the retrieval engine **cannot perform a real budget check yet**. It intentionally marks price as unknown rather than inventing prices.

When price data is added to the product JSON, the same `max_price_inr` argument will automatically begin filtering by budget.

### Next project layer

The next major component should be the **Bundle Optimizer**:

```text
Hybrid Retrieval
       ↓
Candidate products by category
       ↓
Bundle combinations
       ↓
Budget constraint
       ↓
Bathroom-space constraint
       ↓
Compatibility checks
       ↓
Ranking
       ↓
Best Match / Budget Optimized / Premium
```

That optimizer will operate on **complete bathroom bundles**, rather than recommending isolated products.
